# 05 - Training Preliminare Modelli DNN
Addestramento con iperparametri di default di MLP, LSTM e GRU. Ogni modello viene addestrato con 3 seed diversi e si seleziona il migliore (Sezione 3.4 della tesi).

In [21]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/t1dbg'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

## Import

In [ ]:
import os
import pandas as pd
import numpy as np
import json
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
)
import tensorflow as tf
import keras
from keras import layers

from lib.data import load_splits, rescale_data, calculate_metrics, print_results, categorize_glucose, HYPO, HYPER, L_BOUND, U_BOUND
from lib.tf_dnn import (create_model, create_callbacks, train_model, predict_in_batches,
                         print_model_summary, prepare_data)

## Funzioni di utilità
Funzioni per il training con seed multipli, la valutazione e la selezione del miglior modello per ciascuna architettura.

In [36]:
def train(X_train, y_train, X_val, y_val, args, seed):
    """Esegui il training nella main pipeline"""

    def set_seeds(seed):
        """Setta i random seed per riproducibilità  in TensorFlow/Keras."""
        # Set NumPy seed
        np.random.seed(seed)
        # Set TensorFlow seeds
        tf.keras.backend.clear_session()
        tf.random.set_seed(seed)
        tf.keras.utils.set_random_seed(seed)

    set_seeds(seed)

    # Costruisci e addestra il modello
    print(f"\nCreating {args.exp_name.upper()} model with seed {seed}...")
    model = create_model(args.exp_name)
    if seed == args.seed:  # Stampa la summary solo per la prima run
        print_model_summary(model)

    print(f"\nTraining {args.exp_name.upper()} model with seed {seed}...")
    model, history = train_model(
        model,
        X_train,
        y_train,
        X_val,
        y_val,
        args.epochs,
        args.batch_size,
        args.lr,
        args.models_path,
        f"{args.exp_name}_seed_{seed}",
    )

    return model, history

In [37]:
def evaluate_model(model, model_type, eval_set, X_cols, y_cols):
    """Valuta il modello e restituisci i risultati"""
    # Valuta il modello sul validation set
    print(f"\nEvaluating model...")
    eval_set = eval_set.copy()  # Non modificare il set originale
    eval_set["y_pred"] = predict_in_batches(model, eval_set[X_cols], model_type)
    eval_set = eval_set.rename(columns={y_cols[-1]: "target"})
    eval_set = rescale_data(eval_set, ["target", "y_pred"])

    # Selezione le colonne da mostrare nei risultati
    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = eval_set[output_columns]

    return results

In [38]:
def calculate_mae(results):
    """Calcola il MAE dei pazienti dal DataFrame risultante"""
    patient_maes = []
    for patient_id in results["Patient_ID"].unique():
        patient_data = results[results["Patient_ID"] == patient_id]
        mae = np.mean(np.abs(patient_data["target"] - patient_data["y_pred"]))
        patient_maes.append(mae)
    return np.mean(patient_maes)

## Pipeline di addestramento
Per ogni architettura (MLP, LSTM, GRU), si eseguono 3 run con seed diversi (42, 43, 44) e si seleziona il modello con il MAE piu basso sul validation set.

In [ ]:
# Configura il training
class Args:
    def __init__(self):
        self.output_path = "outputs/val_set"
        self.models_path = "models/val_set"
        self.exp_names = ["mlp", "lstm", "gru"]
        self.seed = 42
        self.batch_size = 4096
        self.epochs = 100
        self.lr = 0.01


args = Args()

# Verifica i parametri
print(f"Configurazione:")
print(f"  - Models: {args.exp_names}")
print(f"  - Output path: {args.output_path}")
print(f"  - Models path: {args.models_path}")
print(f"  - Seed: {args.seed}")
print(f"  - Batch size: {args.batch_size}")
print(f"  - Epochs: {args.epochs}")
print(f"  - Learning rate: {args.lr}")

In [ ]:
# Setup ambiente
os.makedirs(args.output_path, exist_ok=True)
os.makedirs(args.models_path, exist_ok=True)

In [ ]:
# Loading dei set dati
train_set, val_set, test_set, X_cols, y_cols = load_splits()

In [ ]:
import shutil

# Training loop per tutti i modelli
for exp_name in args.exp_names:
    print(f"\n{'='*60}")
    print(f"TRAINING {exp_name.upper()}")
    print(f"{'='*60}")

    # Imposta exp_name negli args per le funzioni che lo usano
    args.exp_name = exp_name

    # Preparazione dei dati per tensorflow (reshape diverso per mlp vs lstm/gru)
    X_train, X_val, X_test, y_train, y_val, y_test = prepare_data(
        train_set, val_set, test_set, X_cols, y_cols, exp_name
    )

    # Definisci i seed per eseguire tre training del modello
    seeds = [args.seed, args.seed + 1, args.seed + 2]

    best_mae = float("inf")
    best_model = None
    best_results = None
    best_seed = None

    print(f"\nTRAINING WITH MULTIPLE SEEDS: {seeds}")

    # Esegui il training tre volte con tre seed diversi
    for i, seed in enumerate(seeds, 1):
        print(f"\n{'-'*40}")
        print(f"RUN {i}/3 - SEED {seed}")
        print(f"{'-'*40}")

        # Esegui il training
        model, history = train(X_train, y_train, X_val, y_val, args, seed)

        # Valuta il modello
        results = evaluate_model(model, exp_name, val_set, X_cols, y_cols)

        # Calcola il MAE
        current_mae = calculate_mae(results)
        print(f"\nSeed {seed} - Patient-based MAE: {current_mae:.4f}")

        # Controlla se questo è il miglior modello finora
        if current_mae < best_mae:
            best_mae = current_mae
            best_model = model
            best_results = results
            best_seed = seed
            print(f"New best model found with seed {seed}.")

        print(f"Current best MAE: {best_mae:.4f} (seed {best_seed})")

    # Salva il miglior modello e i risultati
    print(f"\nSAVING BEST {exp_name.upper()} MODEL")
    print(f"Best model achieved with seed {best_seed}")
    print(f"Best MAE: {best_mae:.4f}")

    # Stampa i risultati finali
    print_results(best_results)

    # Salva il miglior modello (rename from temporary seed-specific name)
    temp_model_path = f"{args.models_path}/{exp_name}_seed_{best_seed}.weights.h5"
    final_model_path = f"{args.models_path}/{exp_name}.weights.h5"

    if os.path.exists(temp_model_path):
        shutil.move(temp_model_path, final_model_path)
        print(f"Best model saved to: {final_model_path}")

        # Clean up other temporary model files
        for seed in seeds:
            if seed != best_seed:
                temp_path = f"{args.models_path}/{exp_name}_seed_{seed}.weights.h5"
                if os.path.exists(temp_path):
                    os.remove(temp_path)

    # Salva i migliori risultati
    output_file = f"{args.output_path}/{exp_name}_output.csv"
    best_results.to_csv(output_file, index=False)
    print(f"Best results saved to: {output_file}")
    print(f"Best model trained with seed {best_seed} (MAE: {best_mae:.4f})")

print(f"\nAll DNN training pipelines completed.")